# Fig. S2B, S2C: agreement between statistical methods

Per-lineage agreement for the human pseudotime LRT: density-coloured padj scatters (the converging mass on y=x; permutation x-axis capped to the finite-B floor) + rank-rank hexbins, each with Spearman rho and concordance. S2c is a three-way Venn per lineage. Same layout as `fig1/supp_2_defensibility`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, gaussian_kde

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

FIG = "Fig2/supp_2"
KB = Path(f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling")
LINS = [("excitatory", "Excitatory neurons"), ("inhibitory", "Inhibitory neurons")]
THR = 0.05
DENS_CMAP = "Blues"  # gene density on the diagonal = the methods converge


def bh(p):
    p = np.asarray(p, float)
    ok = np.isfinite(p)
    out = np.full(p.shape, np.nan)
    pp = p[ok]
    n = len(pp)
    o = np.argsort(pp)
    q = np.minimum.accumulate((pp[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    res = np.empty(n)
    res[o] = np.clip(q, 0, 1)
    out[ok] = res
    return out


def density_scatter(ax, xv, yv):
    xv = np.asarray(xv, float)
    yv = np.asarray(yv, float)
    try:
        dens = gaussian_kde(np.vstack([xv, yv]))(np.vstack([xv, yv]))
    except Exception:
        H, xe, ye = np.histogram2d(xv, yv, bins=50)
        ix = np.clip(np.digitize(xv, xe) - 1, 0, H.shape[0] - 1)
        iy = np.clip(np.digitize(yv, ye) - 1, 0, H.shape[1] - 1)
        dens = H[ix, iy]
    o = np.argsort(dens)
    return ax.scatter(
        xv[o],
        yv[o],
        c=dens[o],
        cmap=DENS_CMAP,
        s=4,
        alpha=0.85,
        linewidths=0,
        rasterized=True,
    )


def rank_hexbin(ax, pa, pb):
    rx = pd.Series(np.asarray(pa, float)).rank(method="average").values
    ry = pd.Series(np.asarray(pb, float)).rank(method="average").values
    hb = ax.hexbin(
        rx, ry, gridsize=40, cmap=DENS_CMAP, bins="log", mincnt=1, linewidths=0
    )
    ax.plot([rx.min(), rx.max()], [ry.min(), ry.max()], color="0.5", lw=0.8, zorder=5)
    return hb


def perm_df(lin):
    f = glob.glob(
        str(
            KB
            / "dirmult_results"
            / lin
            / "permutation_batch_number"
            / "dirmult_regression_permutation_*.csv"
        )
    )[0]
    p = (
        pd.read_csv(f).dropna(subset=["p_value_chisq", "p_value_perm"]).copy()
    )  # All tested genes (reproduces the 92.5/88.1 quoted in the text)
    p["padj_chi"] = bh(p.p_value_chisq.values)
    p["padj_perm"] = bh(p.p_value_perm.values)
    return p.dropna(subset=["padj_chi", "padj_perm"])


def pb_df(lin):
    pb = pd.read_csv(
        glob.glob(
            str(
                KB
                / "dirmult_results"
                / lin
                / "pseudobulk_pseudotime"
                / "drimseq_gene_*.csv"
            )
        )[0]
    )
    comp = pd.read_csv(
        KB / "composite_ranking" / f"composite_ranking_optimized2_{lin}.csv"
    )
    keyL = (
        "gene_id"
        if pb["gene_id"].astype(str).str.startswith("ENSG").any()
        else "gene_name"
    )
    return pb.merge(
        comp[[keyL, "padj"]].rename(columns={keyL: "gene_id"}),
        on="gene_id",
        how="inner",
    ).dropna(subset=["adj_pvalue", "padj"])


tw = pd.read_csv(
    KB / "dirmult_results" / "three_way_defensibility_per_lineage.csv"
).set_index("lineage")

In [ ]:
# S2a - permutation vs chi-squared, per lineage (density; x capped to the permutation floor)
fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, S.mm_to_px(80) / 300))
for ax, (lin, lab) in zip(axes, LINS):
    d = perm_df(lin)
    rho = spearmanr(d.p_value_chisq, d.p_value_perm).correlation
    conc = 100 * (((d.padj_chi < THR) == (d.padj_perm < THR)).mean())
    xclip = min(
        -np.log10(float(d.padj_perm[d.padj_perm > 0].min())), 15
    )  # permutation has a finite-B floor
    xv = (
        (-np.log10(d.padj_chi.clip(lower=1e-300))).clip(upper=xclip).values
    )  # chi-sq capped to the same floor
    yv = (-np.log10(d.padj_perm.clip(lower=1e-300))).clip(upper=xclip).values
    sc = density_scatter(ax, xv, yv)
    ax.plot([0, xclip], [0, xclip], color="0.5", lw=0.8, zorder=6)
    t = -np.log10(THR)
    ax.axhline(t, color="k", lw=0.4, ls=":")
    ax.axvline(t, color="k", lw=0.4, ls=":")
    ax.set_xlim(0, xclip * 1.03)
    ax.set_ylim(0, xclip * 1.03)
    ax.text(
        0.04,
        0.97,
        f"Spearman $\\rho$ = {rho:.2f}\nconcordance {conc:.1f}%",
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT - 0.5,
        va="top",
        ha="left",
        fontweight="bold",
    )
    ax.set_title(
        lab,
        fontsize=S.FONT_ANNOT,
        color=S.paper_palette_lineage[lin],
        fontweight="bold",
        pad=3,
    )
    ax.set_xlabel(r"$-\log_{10}$ padj  ($\chi^2$ LRT)", fontsize=S.FONT_LABEL)
    ax.set_ylabel(r"$-\log_{10}$ padj  (permutation)", fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.spines[["top", "right"]].set_visible(False)
fig.text(
    0.5,
    0.01,
    "both axes capped at the finite-B permutation resolution floor; colour = gene density",
    ha="center",
    fontsize=S.FONT_ANNOT - 1.5,
    color="0.5",
    style="italic",
)
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.15, wspace=0.26)
S.save_and_show(fig, "fig_s2a_permutation_vs_chisq", figure=FIG)

In [ ]:
# S2b - pseudobulk DRIMSeq vs cell-level, per lineage (density; the strict subset)
fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, S.mm_to_px(80) / 300))
for ax, (lin, lab) in zip(axes, LINS):
    m = pb_df(lin)
    rho = spearmanr(m.padj, m.adj_pvalue).correlation
    pbo = int(tw.loc[lin, "n_pb_only"])
    ycap = 6.0  # most pseudobulk genes sit near 0; clip the few very-significant ones so the bulk fills the panel
    xv = (-np.log10(m.padj.clip(lower=1e-300))).clip(upper=10).values
    yv = (-np.log10(m.adj_pvalue.clip(lower=1e-300))).clip(upper=ycap).values
    sc = density_scatter(ax, xv, yv)
    ax.plot([0, ycap], [0, ycap], color="0.6", lw=0.7, ls="--", zorder=6)
    ax.axhline(-np.log10(THR), color="k", lw=0.4, ls=":")
    ax.axvline(-np.log10(THR), color="k", lw=0.4, ls=":")
    ax.set_xlim(0, 10)
    ax.set_ylim(0, ycap)
    ax.text(
        0.04,
        0.97,
        f"Spearman $\\rho$ = {rho:.2f}",
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT - 0.5,
        va="top",
        ha="left",
        fontweight="bold",
    )
    ax.text(
        0.97,
        0.04,
        f"{pbo} pseudobulk-only\nof {len(m):,} testable",
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT - 1.5,
        va="bottom",
        ha="right",
        color="0.5",
        style="italic",
    )
    ax.set_title(
        lab,
        fontsize=S.FONT_ANNOT,
        color=S.paper_palette_lineage[lin],
        fontweight="bold",
        pad=3,
    )
    ax.set_xlabel(r"$-\log_{10}$ padj  (cell-level DM LRT)", fontsize=S.FONT_LABEL)
    ax.set_ylabel(r"$-\log_{10}$ padj  (pseudobulk)", fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.13, wspace=0.26)
S.save_and_show(fig, "fig_s2b_pseudobulk_vs_celllevel", figure=FIG)

In [ ]:
# S2c - three-way Venn per lineage (chi-sq / permutation / pseudobulk)
from matplotlib_venn import venn3, venn3_circles

fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, 820 / 300))
for ax, (lin, lab) in zip(axes, LINS):
    r = tw.loc[lin]
    subsets = (
        int(r.n_chisq_only),
        int(r.n_perm_only),
        int(r.n_chisq_and_perm_not_pb),
        int(r.n_pb_only),
        int(r.n_chisq_and_pb_not_perm),
        int(r.n_perm_and_pb_not_chisq),
        int(r.n_all_three),
    )
    v = venn3(
        subsets=subsets,
        set_labels=("chi-sq", "perm", "pseudobulk"),
        ax=ax,
        set_colors=("#2166AC", "#1b7837", "#B2182B"),
        alpha=0.45,
    )
    venn3_circles(subsets=subsets, ax=ax, linewidth=0.6, color="0.4")
    for t in v.set_labels or []:
        if t:
            t.set_fontsize(S.FONT_TICK)
    for t in v.subset_labels or []:
        if t:
            t.set_fontsize(S.FONT_TICK - 1)
    ax.set_title(
        f"{lab}  ({int(r.n_pb_only)} pseudobulk-only of {int(r.n_three_way_tested):,})",
        fontsize=S.FONT_ANNOT - 0.5,
        color=S.paper_palette_lineage[lin],
    )
fig.subplots_adjust(left=0.02, right=0.98, top=0.88, bottom=0.02, wspace=0.05)
S.save_and_show(fig, "fig_s2c_three_way_venn", figure=FIG)

In [ ]:
# S2d - rank-rank concordance, permutation vs chi-squared, per lineage (hexbin density)
fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, S.mm_to_px(80) / 300))
for ax, (lin, lab) in zip(axes, LINS):
    d = perm_df(lin)
    rho = spearmanr(d.p_value_chisq, d.p_value_perm).correlation
    hb = rank_hexbin(ax, d.p_value_chisq, d.p_value_perm)
    ax.text(
        0.04,
        0.97,
        f"Spearman $\\rho$ = {rho:.2f}",
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT - 0.5,
        va="top",
        ha="left",
        fontweight="bold",
    )
    ax.set_title(
        lab,
        fontsize=S.FONT_ANNOT,
        color=S.paper_palette_lineage[lin],
        fontweight="bold",
        pad=3,
    )
    ax.set_xlabel(r"gene rank  ($\chi^2$ LRT)", fontsize=S.FONT_LABEL)
    ax.set_ylabel("gene rank  (permutation)", fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.13, wspace=0.26)
S.save_and_show(fig, "fig_s2d_rank_permutation", figure=FIG)

In [ ]:
# S2e - rank-rank concordance, pseudobulk vs cell-level, per lineage (hexbin density)
fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, S.mm_to_px(80) / 300))
for ax, (lin, lab) in zip(axes, LINS):
    m = pb_df(lin)
    rho = spearmanr(m.padj, m.adj_pvalue).correlation
    hb = rank_hexbin(ax, m.padj, m.adj_pvalue)
    ax.text(
        0.04,
        0.97,
        f"Spearman $\\rho$ = {rho:.2f}",
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT - 0.5,
        va="top",
        ha="left",
        fontweight="bold",
    )
    ax.set_title(
        lab,
        fontsize=S.FONT_ANNOT,
        color=S.paper_palette_lineage[lin],
        fontweight="bold",
        pad=3,
    )
    ax.set_xlabel("gene rank  (cell-level DM LRT)", fontsize=S.FONT_LABEL)
    ax.set_ylabel("gene rank  (pseudobulk)", fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.13, wspace=0.26)
S.save_and_show(fig, "fig_s2e_rank_pseudobulk", figure=FIG)

## S2a + S2c size variants (half width; quarter/eighth heights)


In [ ]:
# --- S2a + S2c size variants (half width; quarter/eighth heights) ---
from matplotlib.ticker import MaxNLocator


def _build_s2a(figsize, tiny):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, (lin, lab) in zip(axes, LINS):
        d = perm_df(lin)
        rho = spearmanr(d.p_value_chisq, d.p_value_perm).correlation
        conc = 100 * (((d.padj_chi < THR) == (d.padj_perm < THR)).mean())
        xclip = min(-np.log10(float(d.padj_perm[d.padj_perm > 0].min())), 15)
        xv = (-np.log10(d.padj_chi.clip(lower=1e-300))).clip(upper=xclip).values
        yv = (-np.log10(d.padj_perm.clip(lower=1e-300))).clip(upper=xclip).values
        density_scatter(ax, xv, yv)
        ax.plot([0, xclip], [0, xclip], color="0.5", lw=0.7, zorder=6)
        t = -np.log10(THR)
        ax.axhline(t, color="k", lw=0.4, ls=":")
        ax.axvline(t, color="k", lw=0.4, ls=":")
        ax.set_xlim(0, xclip * 1.03)
        ax.set_ylim(0, xclip * 1.03)
        txt = (
            f"$\\rho$ = {rho:.2f}"
            if tiny
            else f"Spearman $\\rho$ = {rho:.2f}\nconcordance {conc:.1f}%"
        )
        ax.text(
            0.05,
            0.96,
            txt,
            transform=ax.transAxes,
            fontsize=S.FONT_ANNOT - 1 if tiny else S.FONT_ANNOT - 0.5,
            va="top",
            ha="left",
            fontweight="bold",
        )
        ax.set_title(
            lab.split()[0][:3].upper(),
            fontsize=S.FONT_ANNOT,
            color=S.paper_palette_lineage[lin],
            fontweight="bold",
            pad=2,
        )
        ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.tick_params(labelsize=S.FONT_TICK)
        ax.spines[["top", "right"]].set_visible(False)
        _fl = S.FONT_TICK if tiny else S.FONT_LABEL
        ax.set_xlabel(
            r"$-\log_{10}$p $\chi^2$" if tiny else r"$-\log_{10}$padj $\chi^2$",
            fontsize=_fl,
        )
        if not tiny or ax is axes[0]:
            ax.set_ylabel(
                r"$-\log_{10}$p perm" if tiny else r"$-\log_{10}$padj perm",
                fontsize=_fl,
            )
    if tiny:
        fig.subplots_adjust(left=0.13, right=0.98, top=0.88, bottom=0.32, wspace=0.30)
    else:
        fig.subplots_adjust(left=0.11, right=0.98, top=0.91, bottom=0.17, wspace=0.30)
    return fig


S.save_and_show(
    _build_s2a((S.HALF_W_IN, S.QUARTER_H_IN), False),
    "fig_s2a_permutation_vs_chisq__quarter_half",
    figure=FIG,
)
S.save_and_show(
    _build_s2a((S.HALF_W_IN, S.EIGHTH_H_IN), True),
    "fig_s2a_permutation_vs_chisq__eighth_half",
    figure=FIG,
)

from matplotlib_venn import venn3, venn3_circles


def _build_s2c(figsize):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, (lin, lab) in zip(axes, LINS):
        r = tw.loc[lin]
        subsets = (
            int(r.n_chisq_only),
            int(r.n_perm_only),
            int(r.n_chisq_and_perm_not_pb),
            int(r.n_pb_only),
            int(r.n_chisq_and_pb_not_perm),
            int(r.n_perm_and_pb_not_chisq),
            int(r.n_all_three),
        )
        v = venn3(
            subsets=subsets,
            set_labels=("chi-sq", "perm", "pb"),
            ax=ax,
            set_colors=("#2166AC", "#1b7837", "#B2182B"),
            alpha=0.45,
        )
        venn3_circles(subsets=subsets, ax=ax, linewidth=0.5, color="0.4")
        for t in v.set_labels or []:
            if t:
                t.set_fontsize(S.FONT_TICK - 1)
        for t in v.subset_labels or []:
            if t:
                t.set_fontsize(S.FONT_TICK - 1)
        ax.set_title(
            lab.split()[0][:3].upper(),
            fontsize=S.FONT_ANNOT,
            color=S.paper_palette_lineage[lin],
            pad=1,
        )
    fig.subplots_adjust(left=0.01, right=0.99, top=0.86, bottom=0.02, wspace=0.04)
    return fig


S.save_and_show(
    _build_s2c((S.HALF_W_IN, S.EIGHTH_H_IN)),
    "fig_s2c_three_way_venn__eighth_half",
    figure=FIG,
)

## S2a (square, quarter-half) + S2c (quarter-half) variants


In [ ]:
# --- S2a square (quarter-half) + S2c (quarter-half) variants ---
from matplotlib.ticker import MaxNLocator


# S2a: square scatter (equal x/y scale) - the standard way to show a 2-variable correlation
def _build_s2a_square(figsize):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, (lin, lab) in zip(axes, LINS):
        d = perm_df(lin)
        rho = spearmanr(d.p_value_chisq, d.p_value_perm).correlation
        conc = 100 * (((d.padj_chi < THR) == (d.padj_perm < THR)).mean())
        xclip = min(-np.log10(float(d.padj_perm[d.padj_perm > 0].min())), 15)
        xv = (-np.log10(d.padj_chi.clip(lower=1e-300))).clip(upper=xclip).values
        yv = (-np.log10(d.padj_perm.clip(lower=1e-300))).clip(upper=xclip).values
        density_scatter(ax, xv, yv)
        ax.plot([0, xclip], [0, xclip], color="0.5", lw=0.7, zorder=6)
        t = -np.log10(THR)
        ax.axhline(t, color="k", lw=0.4, ls=":")
        ax.axvline(t, color="k", lw=0.4, ls=":")
        ax.set_xlim(0, xclip * 1.03)
        ax.set_ylim(0, xclip * 1.03)
        ax.set_aspect("equal", adjustable="box")
        ax.text(
            0.05,
            0.96,
            f"Spearman $\\rho$ = {rho:.2f}\nconc. {conc:.1f}%",
            transform=ax.transAxes,
            fontsize=S.FONT_ANNOT - 1,
            va="top",
            ha="left",
            fontweight="bold",
        )
        ax.set_title(
            lab.split()[0][:3].upper(),
            fontsize=S.FONT_ANNOT,
            color=S.paper_palette_lineage[lin],
            fontweight="bold",
            pad=2,
        )
        ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.tick_params(labelsize=S.FONT_TICK)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel(r"$-\log_{10}$padj $\chi^2$", fontsize=S.FONT_LABEL)
        if ax is axes[0]:
            ax.set_ylabel(r"$-\log_{10}$padj perm", fontsize=S.FONT_LABEL)
    fig.subplots_adjust(left=0.12, right=0.98, top=0.90, bottom=0.16, wspace=0.35)
    return fig


S.save_and_show(
    _build_s2a_square((S.HALF_W_IN, S.QUARTER_H_IN)),
    "fig_s2a_permutation_vs_chisq__quarter_half_square",
    figure=FIG,
)

# S2c quarter-half: bigger venns, tight horizontal spacing (small EXC<->INH gap)
from matplotlib_venn import venn3, venn3_circles


def _build_s2c_q(figsize):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, (lin, lab) in zip(axes, LINS):
        r = tw.loc[lin]
        subsets = (
            int(r.n_chisq_only),
            int(r.n_perm_only),
            int(r.n_chisq_and_perm_not_pb),
            int(r.n_pb_only),
            int(r.n_chisq_and_pb_not_perm),
            int(r.n_perm_and_pb_not_chisq),
            int(r.n_all_three),
        )
        v = venn3(
            subsets=subsets,
            set_labels=("chi-sq", "perm", "pb"),
            ax=ax,
            set_colors=("#2166AC", "#1b7837", "#B2182B"),
            alpha=0.45,
        )
        venn3_circles(subsets=subsets, ax=ax, linewidth=0.5, color="0.4")
        for t in v.set_labels or []:
            if t:
                t.set_fontsize(S.FONT_TICK)
        for t in v.subset_labels or []:
            if t:
                t.set_fontsize(S.FONT_TICK - 1)
        ax.set_title(
            lab.split()[0][:3].upper(),
            fontsize=S.FONT_ANNOT,
            color=S.paper_palette_lineage[lin],
            pad=2,
        )
    fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.02, wspace=0.02)
    return fig


S.save_and_show(
    _build_s2c_q((S.HALF_W_IN, S.QUARTER_H_IN)),
    "fig_s2c_three_way_venn__quarter_half",
    figure=FIG,
)